## Experimento 3 — Inclusão da Variável Nitrogênio Sem Balanceamento

Este experimento investiga o impacto da adição da variável **Nitrogênio (`Nitrogen (mg/l)`)** no desempenho de modelos de classificação para a qualidade da água (`conama_status`). Nos testes anteriores — utilizando apenas `Temperature (cel)`, `Orthophosphate (mg/l)`, `Country` e `Waterbody Type` —, os modelos apresentaram forte tendência de superclassificação na classe "Excelente" e dificuldades para reconhecer classes intermediárias e críticas, devido ao desbalanceamento do *dataset* e à complexidade do problema.

Embora o nitrogênio **não tenha sido utilizado diretamente na criação do rótulo alvo** (por falta de equivalência direta com as especificações da Resolução CONAMA), sua inclusão justifica-se por sua relevância ambiental indireta. Na dinâmica hídrica, níveis de nitrogênio atuam como indicadores (*proxies*) de fenômenos complexos, como o aporte de matéria orgânica, uso de fertilizantes, eutrofização e degradação da qualidade da água. O objetivo é avaliar se os algoritmos conseguem capturar essas relações ocultas e associações matemáticas para melhorar a generalização e reduzir os erros de classificação.

### Metodologia e Abordagem Comparativa

Para isolar o impacto da nova variável, o *dataset* foi mantido sem técnicas de balanceamento. A base foi dividida em **80% para treino e 20% para teste**, utilizando `train_test_split` com `random_state=42` e divisão estratificada (`stratify=y`), replicando exatamente a estrutura dos experimentos anteriores.

A análise central deste experimento baseia-se no **confronto de desempenho entre os dois primeiros algoritmos (Random Forest e LightGBM) ao lidarem com o mesmo incremento de informação:

Essa abordagem comparativa permite identificar qual das duas arquiteturas possui maior sensibilidade para extrair os padrões indiretos do nitrogênio, mitigando a superclassificação da classe majoritária sem a necessidade de balanceamento sintético. O contraste entre os modelos será sustentado pela análise cruzada das métricas de *Accuracy*, *Precision*, *Recall*, *F1-score* e pelo comportamento detalhado de suas respectivas Matrizes de Confusão.

## Preparação do ambiente

In [1]:
# IMPORT DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

from lightgbm import LGBMClassifier

## Definição de X e y

Em relação ao Experimento 1, a única mudança é a adição da variável
`Nitrogen (mg/l)` ao conjunto de entrada, passando de 4 para 5
variáveis. As demais condições permanecem idênticas.

In [4]:
X = df[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type",
    "Nitrogen (mg/l)"
]]

y = df["conama_status"]

In [5]:
# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 5)
Teste: (28280, 5)


In [6]:
# Pré-processamento
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [7]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [8]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 LGBMClassifier(n_jobs=-1, random_state=42, verbose=-1))])

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi
realizada a avaliação das métricas no conjunto de treino, permitindo
comparar o comportamento entre treino e teste e identificar possíveis
sinais de overfitting.

In [9]:
# Métricas de treino
y_train_pred = model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_confusion_matrix = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.7922276540634199
Train Precision:
0.7665607452847399
Train Recall:
0.7922276540634199
Train F1:
0.7696619001748355
Train Confusion Matrix:
[[ 1994  2630    47  2889]
 [  759  8993    40 11886]
 [  161   555   152   238]
 [  573  3695    30 78477]]


In [10]:
# Métricas de teste
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7824257425742575

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.50      0.23      0.31      1890
         Boa       0.54      0.40      0.46      5419
     Crítica       0.19      0.05      0.08       277
   Excelente       0.84      0.94      0.89     20694

    accuracy                           0.78     28280
   macro avg       0.52      0.40      0.43     28280
weighted avg       0.75      0.78      0.76     28280


Confusion Matrix:
[[  431   698    26   735]
 [  216  2154    22  3027]
 [   41   140    13    83]
 [  175   983     7 19529]]


## Resultados Obtidos — Experimento 3

A acurácia obtida no conjunto de teste foi de aproximadamente:

```python
0.782
```

Enquanto isso, a acurácia no conjunto de treino ficou em torno de:

```python
0.792
```

### Comparação com Experimento 3
*LightGBM 4 variáveis vs LightGBM 5 variáveis — sem balanceamento*

| Métrica             | Exp 1 — LGBM 4 var | Exp 3 — LGBM 5 var |
|---------------------|--------------------|--------------------|
| Accuracy treino     | 0.750              | 0.792              |
| Accuracy teste      | 0.745              | 0.782              |
| Overfitting         | 0.005              | 0.010              |
| Precision Atenção   | 0.42               | 0.50               |
| Precision Boa       | 0.39               | 0.54               |
| Precision Crítica   | 0.17               | 0.19               |
| Precision Excelente | 0.79               | 0.84               |
| Recall Atenção      | 0.21               | 0.23               |
| Recall Boa          | 0.16               | 0.40               |
| Recall Crítica      | 0.01               | 0.05               |
| Recall Excelente    | 0.96               | 0.94               |
| F1 Atenção          | 0.28               | 0.31               |
| F1 Boa              | 0.23               | 0.46               |
| F1 Crítica          | 0.02               | 0.08               |
| F1 Excelente        | 0.87               | 0.89               |

### Comparação entre algoritmos — 5 variáveis sem balanceamento
*Random Forest (Experimento 3) vs LightGBM (Experimento 3)*

| Métrica             | RF 5 var s/ bal | LGBM 5 var s/ bal |
|---------------------|-----------------|-------------------|
| Accuracy treino     | 0.932           | 0.792             |
| Accuracy teste      | 0.765           | 0.782             |
| Overfitting         | 0.167           | 0.010             |
| Precision Atenção   | 0.41            | 0.50              |
| Precision Boa       | 0.50            | 0.54              |
| Precision Crítica   | 0.16            | 0.19              |
| Precision Excelente | 0.84            | 0.84              |
| Recall Atenção      | 0.24            | 0.23              |
| Recall Boa          | 0.42            | 0.40              |
| Recall Crítica      | 0.06            | 0.05              |
| Recall Excelente    | 0.91            | 0.94              |
| F1 Atenção          | 0.30            | 0.31              |
| F1 Boa              | 0.46            | 0.46              |
| F1 Crítica          | 0.09            | 0.08              |
| F1 Excelente        | 0.87            | 0.89              |

### Conclusão

## Experimento 3 — LGBM 5 variáveis sem balanceamento

A adição de uma quinta variável ao modelo LightGBM sem balanceamento produziu ganhos modestos porém consistentes em relação ao Experimento 1, com melhora generalizada nas classes minoritárias sem deterioração da capacidade de generalização.

Em `Crítica`, a classe de maior dificuldade na sequência experimental, o avanço foi visível: precision subiu de 0.17 para 0.19 e recall de 0.01 para 0.05, elevando o F1 de 0.02 para 0.08. Ainda que os valores absolutos permaneçam baixos, o salto relativo no recall — cinco vezes maior — indica que a variável adicionada trouxe informação relevante para distinguir amostras críticas. No contexto de qualidade hídrica, qualquer aumento na detecção dessa classe tem peso desproporcional frente aos demais erros.

`Atenção` apresentou o ganho mais expressivo em recall entre as classes minoritárias: de 0.21 para 0.23, com precision subindo de 0.42 para 0.50 — um movimento raro na sequência experimental, em que ambas as métricas avançaram simultaneamente. O F1 subiu de 0.28 para 0.31, consolidando a melhora.

`Boa` registrou o avanço mais substantivo em termos absolutos: recall dobrou de 0.16 para 0.40 e precision subiu de 0.39 para 0.54, resultando em F1 de 0.46 ante 0.23. Esse salto sugere que a quinta variável é particularmente discriminativa para amostras dessa classe — o modelo passou a identificar com muito mais frequência e precisão as condições classificadas como boas.

`Excelente` manteve comportamento estável: precision ficou em 0.84, recall recuou levemente de 0.96 para 0.94, e F1 subiu de 0.87 para 0.89. A classe majoritária continua dominando o desempenho geral, mas sem ampliar a distorção em relação às minoritárias.

A acurácia de treino subiu de 0.750 para 0.792 e a de teste de 0.745 para 0.782, com overfitting praticamente estável — de 0.005 para 0.010 —, confirmando que a variável adicional aumentou a capacidade preditiva sem comprometer a generalização.

---

## Comparação RF vs LGBM — 5 variáveis sem balanceamento

A comparação direta entre os dois algoritmos na mesma configuração expõe dinâmicas opostas de aprendizado: o Random Forest concentrou seu desempenho na classe majoritária e memorizou o treino; o LightGBM distribuiu melhor o aprendizado entre as classes com generalização superior.

O overfitting é o indicador mais revelador dessa diferença: 0.167 no Random Forest contra 0.010 no LightGBM. O RF atingiu acurácia de treino de 0.932, mas entregou apenas 0.765 no teste — evidência clara de memorização. O LightGBM, com treino em 0.792, chegou a 0.782 no teste, demonstrando que o que aprendeu é transferível para dados novos.

Em `Crítica`, o LightGBM foi superior em todas as métricas: precision de 0.19 contra 0.16, recall de 0.05 contra 0.06 — uma diferença marginal em recall, mas com F1 de 0.08 ante 0.09 do RF, classe onde o volume de amostras é tão baixo que qualquer acerto ou erro move os indicadores de forma desproporcional.

`Atenção` foi o diferencial mais claro fora de `Excelente`: o LGBM entregou precision de 0.50 e recall de 0.23 ante 0.41 e 0.24 do RF. O F1 do LGBM ficou em 0.31 contra 0.30 do RF — vantagem pequena, mas obtida com precision consideravelmente mais alta, indicando menos falsos positivos.

`Boa` foi a classe onde o LightGBM mostrou maior superioridade: recall de 0.40 contra 0.42 do RF — desempenho equivalente em detecção — mas precision de 0.54 contra 0.50, resultando em F1 de 0.46 para ambos. O LGBM identificou amostras `Boa` com a mesma frequência e maior precisão.

`Excelente` foi a única classe em que o Random Forest foi competitivo: recall de 0.91 contra 0.94 do LGBM, com F1 de 0.87 para ambos. Esse resultado, porém, é em parte produto do viés do RF para a classe majoritária — favorecimento que se reflete no overfitting elevado e na queda de desempenho nas demais classes.

O balanço geral favorece o LightGBM como algoritmo mais robusto nessa configuração: generaliza melhor, distribui o aprendizado com mais equidade entre as classes e entrega desempenho superior nas categorias de maior relevância operacional para monitoramento de qualidade hídrica.

In [ ]:
test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average="weighted")

resultados = {
    "experimento": "exp_03_lightgbm_5var_sem_balanceamento",
    "algoritmo": "LightGBM",
    "balanceamento": "nenhum",
    "n_features": X.shape[1],
    "accuracy_treino": round(train_accuracy, 4),
    "accuracy_teste": round(test_accuracy, 4),
    "f1_weighted_treino": round(train_f1, 4),
    "f1_weighted_teste": round(test_f1, 4),
}

pd.DataFrame([resultados]).to_csv(
    "/content/drive/MyDrive/EDA_AquaSense/resultados_experimentos.csv",
    mode="a",
    index=False,
    header=False
)

print("Métricas salvas.")

Métricas salvas.
